In [1]:
import hashlib
import secrets

def hash_password(password):
    salt = secrets.token_hex(16)
    password_hash = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt.encode("utf-8"),
        200_000
    ).hex()
    return salt, password_hash

def verify_password(password, salt, stored_hash):
    test_hash = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt.encode("utf-8"),
        200_000
    ).hex()
    return secrets.compare_digest(test_hash, stored_hash)

# Test password security
password = "DemoPassword123!"
salt, stored_hash = hash_password(password)

print("Password hash created:", bool(stored_hash))
print("Correct password accepted:",
      verify_password(password, salt, stored_hash))
print("Wrong password rejected:",
      not verify_password("WrongPassword", salt, stored_hash))

Password hash created: True
Correct password accepted: True
Wrong password rejected: True


In [2]:

from flask import Flask, request, session
from werkzeug.security import generate_password_hash, check_password_hash
import sqlite3
import secrets
import os

app = Flask(__name__)
app.config["SECRET_KEY"] = secrets.token_hex(32)

DB_PATH = "/content/users.db"

# Create the user database
def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL
            )
        """)

@app.route("/")
def home():
    return "Secure Web Application is running!"

@app.route("/register", methods=["POST"])
def register():
    username = request.form.get("username", "").strip()
    password = request.form.get("password", "")

    if not username or not password:
        return "Username and password are required.", 400

    password_hash = generate_password_hash(password)

    try:
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "INSERT INTO users VALUES (?, ?)",
                (username, password_hash)
            )
        return "Registration successful!"
    except sqlite3.IntegrityError:
        return "Username already exists.", 409

@app.route("/login", methods=["POST"])
def login():
    username = request.form.get("username", "").strip()
    password = request.form.get("password", "")

    with sqlite3.connect(DB_PATH) as conn:
        user = conn.execute(
            "SELECT password_hash FROM users WHERE username = ?",
            (username,)
        ).fetchone()

    if user and check_password_hash(user[0], password):
        session["username"] = username
        return "Login successful!"

    return "Invalid username or password.", 401

@app.route("/dashboard")
def dashboard():
    if "username" not in session:
        return "Access denied. Please log in.", 401

    return "Welcome to your protected dashboard!"

@app.route("/logout", methods=["POST"])
def logout():
    session.clear()
    return "Logged out successfully."

# Run security tests
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

init_db()

with app.test_client() as client:
    print("Test 1: Registration")
    response = client.post("/register", data={
        "username": "demo_user",
        "password": "DemoPassword123!"
    })
    print(response.status_code, response.get_data(as_text=True))

    print("\nTest 2: Correct password")
    response = client.post("/login", data={
        "username": "demo_user",
        "password": "DemoPassword123!"
    })
    print(response.status_code, response.get_data(as_text=True))

    print("\nTest 3: Protected dashboard")
    response = client.get("/dashboard")
    print(response.status_code, response.get_data(as_text=True))

    print("\nTest 4: Logout")
    response = client.post("/logout")
    print(response.status_code, response.get_data(as_text=True))

    print("\nTest 5: Dashboard after logout")
    response = client.get("/dashboard")
    print(response.status_code, response.get_data(as_text=True))


Test 1: Registration
200 Registration successful!

Test 2: Correct password
200 Login successful!

Test 3: Protected dashboard
200 Welcome to your protected dashboard!

Test 4: Logout
200 Logged out successfully.

Test 5: Dashboard after logout
401 Access denied. Please log in.


## Security Testing Results

The application was tested using Flask's test client.

| Test | Expected Result | Status |
|---|---|---|
| User registration | Registration successful | PASS |
| Correct password | Login successful | PASS |
| Protected dashboard | Authorized access allowed | PASS |
| Logout | Session cleared | PASS |
| Dashboard after logout | Access denied | PASS |

## Security Features
- Passwords are stored as secure password hashes instead of plain text.
- Parameterized SQL queries help prevent SQL injection.
- Login credentials are verified using Werkzeug security utilities.
- Session-based authentication protects the dashboard.
- Logout clears the user's session.

## Limitations
This is an educational prototype tested in Google Colab. It is not a production-ready or publicly hosted website. Production deployment would require additional protections, including CSRF protection, HTTPS, secure cookie settings, rate limiting, and stronger input validation.

# Secure Web Application

## Project Overview
This project demonstrates basic web application security using Python. It focuses on secure password storage, user authentication, and access control.

## Objectives
- Implement secure password hashing.
- Validate user login credentials.
- Protect pages from unauthenticated access.
- Demonstrate basic security testing.

## Technologies Used
- Python
- Flask
- SQLite
- Werkzeug password hashing utilities
- Google Colab

## Disclaimer
This project is for educational purposes only. It is not intended for production deployment.